In [ ]:
# Data libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def reshape_long(avg_df, min_df, max_df, loc_df, start = "2017-04-01", drop_tidal = True):
    
    data = [avg_df, min_df, max_df]  
    label = ["avg", "min", "max"]
    column = ["County", "Station_Name", "Observation_ID", "date"]
    long_df = {}

    for df, lab in zip(data, label):
        if drop_tidal:
            df = df[~df["Observation_ID"].str.contains("tidal-level", na = False)]  
        
        long_df[lab] = df.melt(id_vars = column[0:3], var_name = "date", value_name = lab)
        long_df[lab]["date"] = pd.to_datetime(long_df[lab]["date"], format = "%d/%m/%Y")
        long_df[lab] = long_df[lab][long_df[lab]["date"] >= start]

    river_long = (long_df["avg"]
                 .merge(long_df["min"], on = column, how = "left")
                 .merge(long_df["max"], on = column, how = "left")
                 .merge(loc_df, on = column[0:3], how = "left"))

    for col in column[0:3]:
        river_long[col] = river_long[col].astype("category")
    
    return river_long

In [ ]:
def clean_error(df, k = 20):
    
    label = ["avg", "min", "max"]
    df2 = df.copy()

    err = (df2[label].isin([99999.999, -999999.999, -99]) | (df2[label].abs() >= 9999)).any(axis = 1)
    df2.attrs["error"] = int(err.sum())
    df2.loc[err, label] = np.nan

    for lab in label:
        per_sta = df2.groupby("Observation_ID", observed = True)[lab]
        
        q1 = per_sta.transform(lambda s: s.quantile(0.25))
        q3 = per_sta.transform(lambda s: s.quantile(0.75))
        iqr = q3 - q1
        
        outlier = ((df2[lab] > q3 + k * iqr) | (df2[lab] < q1 - k * iqr))
        df2.attrs[f"outliers_{lab}"] = int(outlier.sum())        
        df2.loc[outlier, lab] = np.nan
    
    return df2

In [ ]:
def drop_ids(df, min_years = 3):
    years = df["avg"].notna().groupby(df["Observation_ID"], observed = True).sum() / 365.25

    keep_ids = years[years >= min_years].index

    keep_df = df[df["Observation_ID"].isin(keep_ids)].copy()
    keep_df.attrs["dropped_ids"] = int((years < min_years).sum())
    keep_df.attrs["kept_ids"] = int((years >= min_years).sum())

    return keep_df

In [ ]:
def make_split(clean_df, cutoff = "2026-08-31",
               feature_cols = ["Latitude", "Longitude"], target = "avg"):

    cutoff = pd.Timestamp(cutoff)

    # split by cutoff date
    train = clean_df[clean_df["date"] <= cutoff]
    test  = clean_df[clean_df["date"] > cutoff]

    # drop rows with no target, then sort + renumber
    train = train.dropna(subset = [target]).sort_values(["Observation_ID", "date"]).reset_index(drop = True)
    test  = test.dropna(subset = [target]).sort_values(["Observation_ID", "date"]).reset_index(drop = True)

    X_train, y_train = train[feature_cols], train[target]
    X_test,  y_test  = test[feature_cols],  test[target]

    return X_train, y_train, X_test, y_test

In [ ]:
# Load data

river_avg = pd.read_csv("data/avg_river_levels.csv")
river_min = pd.read_csv("data/min_river_levels.csv")
river_max = pd.read_csv("data/max_river_levels.csv")

river_location = pd.read_csv("data/monitoring_locations.csv")

In [ ]:
# Data cleaning 

river_long = reshape_long(river_avg, river_min, river_max, river_location)
river_long2 = clean_error(river_long)
river_long3 = drop_ids(river_long2)

In [ ]:
# Split data 

X_train, y_train, X_test, y_test = make_split(river_long3)

In [ ]:
# Check

X_train.shape, y_train.shape

In [ ]:
X_test.shape, y_test.shape